# Teach it what NOT to say: preference tuning with DPO

Your Yoda adapter learned from demonstrations. Every demonstration in
`dvgodoy/yoda_sentences` came with a habit attached: the trailing "Yes, hrrmm."
interjections of the `translation_extra` column. The model learned the inversion you
wanted and the tic you never asked for, because supervised fine-tuning cannot tell them
apart. A demonstration says "produce this". It cannot say "this part, not that part."

Preference pairs can. [Direct Preference Optimization (DPO)](https://proceedings.neurips.cc/paper_files/paper/2023/hash/a85b405ed65c6477a4fe8302b5e06ce7-Abstract-Conference.html) trains on a **contrast**: the
same prompt with a chosen and a rejected response. The only new information is the
difference between them. Today the difference is the interjections. The question is
whether we can remove the habit without paying for the fix twice.

## Three axes

| Axis | Measured on | Metric |
| --- | --- | --- |
| **Habit** | 20 held-out sentences (`yoda_external_test.csv`) | interjection rate |
| **Style** | the same 20 sentences, clean Yoda references | BERTScore F1, chrF |
| **Capability** | `phi3_capability_probe.csv` (calibrated) | probe accuracy |

Last module you measured what SFT cost you. This module asks whether preference
training levies its own tax while it fixes the habit.

## Before you start

- Runtime &rarr; Change runtime type &rarr; **T4 GPU**.
- Upload `yoda_external_test.csv`, `phi3_capability_probe.csv`, and your trained Yoda
  adapter folder (from Chapter 0's `trainer.save_model(...)`).
- The base model, quantization config, and decoding settings match the earlier
  notebooks exactly, so the numbers stay comparable. Do not change them.

*This module builds on Daniel Voigt Godoy's Chapter 0 QLoRA notebook and his
`dvgodoy/yoda_sentences` dataset.*

## Stated hypothesis

**Hypothesis: preference tuning removes the habit without erasing the style.**
DPO trains on pairs that differ only in the trailing "Yes, hrrmm." interjections. It should cut the habit rate on
held-out data while reference similarity stays near its pre-DPO value.

Before running anything, write down your expectation: how far can the habit rate fall
before the style degrades? The before/after table tests the hypothesis. A null result,
honestly obtained and clearly reported, is a success.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 evaluate bert-score sacrebleu pandas matplotlib

In [ ]:
import os
import re
import string

import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset, load_dataset
from peft import PeftModel, prepare_model_for_kbit_training
from sacrebleu import corpus_chrf
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

torch.manual_seed(42)

# ---- Paths and settings ------------------------------------------------
REPO_ID = "microsoft/Phi-3-mini-4k-instruct"
COURSE_DIR = "/content/drive/MyDrive/344/344-FA26-Students/344-FA26-code"  # TODO: your Drive folder
ADAPTER_PATH = f"{COURSE_DIR}/local-phi3-mini-yoda-adapter"  # or a Hugging Face repo id
STYLE_CSV = f"{COURSE_DIR}/yoda_external_test.csv"
PROBE_CSV = f"{COURSE_DIR}/phi3_capability_probe.csv"
DPO_OUTPUT_DIR = "/content/phi3-mini-yoda-dpo-adapter"
RESULTS_PATH = f"{COURSE_DIR}/yoda_dpo_results.csv"

BETA = 0.1              # DPO's one new hyperparameter. Sweep it as an extension.
MAX_NEW_TOKENS = 64     # same as every earlier notebook
HELDOUT_PAIRS = 40      # pairs reserved for the post-training margin check
# ------------------------------------------------------------------------

assert os.path.isfile(STYLE_CSV), f"Upload the style test set to {STYLE_CSV}"
assert os.path.isfile(PROBE_CSV), f"Upload the capability probe to {PROBE_CSV}"

YODA_SYSTEM_PROMPT = """
You rewrite ordinary English sentences in Yoda's distinctive speaking style.

Rules:
- Preserve the original meaning.
- Use Yoda-like inverted sentence structure.
- Keep the response concise.
- You may occasionally use expressions such as "Hmm" or "Hrrmm."
- Return only the rewritten sentence.
- Do not explain your answer.
""".strip()

In [ ]:
# Identical configuration to the evaluation and alignment-tax notebooks - do not
# change, or the numbers stop being comparable across notebooks.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    device_map="auto",
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

print(f"Model memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
style_df = pd.read_csv(STYLE_CSV)
probe_df = pd.read_csv(PROBE_CSV)

assert {"sentence", "translation_extra"}.issubset(style_df.columns)
assert {"prompt", "check_type", "expected", "category"}.issubset(probe_df.columns)

style_sentences = style_df["sentence"].tolist()
style_references = style_df["translation_extra"].tolist()

print(f"Style test set:     {len(style_df)} sentences")
print(f"Capability probe:   {len(probe_df)} items")
print(probe_df["category"].value_counts().to_string())

## The habit detector

DPO can only remove what a checker can see, so the checker comes first and proves
itself on known cases before any pair is built or any GPU minute is spent. The same
rule as last module: never trust a scorer you have not tested.

One caution. The pattern treats a bare "yes" as an interjection. That is right for
this dataset, where the tic is appended to otherwise clean sentences, but it would be
wrong for text where "yes" carries meaning. A checker is a measurement instrument
with a valid range.

In [ ]:
INTERJECTION_RE = re.compile(r"\b(?:ye+s+|yr+s+|hr+m+|hm+m*|mm+)\b", re.IGNORECASE)


def has_interjection(text):
    return bool(INTERJECTION_RE.search(text))


def interjection_rate(texts):
    return float(np.mean([has_interjection(t) for t in texts]))


# Self-test on known positives and known near-misses before trusting it anywhere.
_cases = [
    ("Yes, hrrmm.", True),
    ("Yesss, finished the prototype, our team has.", True),
    ("Finished the prototype, our team has. Yrsssss.", True),
    ("Strong you are. Hmm.", True),
    ("Hrmmm. Ready, you are not.", True),
    ("Yesterday, to the market she went.", False),
    ("Before lunch, the lecture slides the professor uploaded.", False),
    ("A hymn, the choir sang.", False),
]
for t, want in _cases:
    assert has_interjection(t) == want, f"checker failed on: {t!r}"
print("interjection checker self-test passed")

In [ ]:
# Copied verbatim from the alignment-tax notebook so probe scoring is identical.
def _norm(text):
    text = text.lower().strip()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def score_probe(output, check_type, expected):
    """True if the model's output satisfies this probe item.

    exact        - normalised output equals expected
    keyword      - any '|'-separated alternative appears
    all_keywords - every ';'-separated term appears
    word_count   - output has exactly N words
    numeric      - expected appears among the numbers in the output
    """
    low = output.lower()
    if check_type == "exact":
        return _norm(output) == _norm(expected)
    if check_type == "keyword":
        return any(alt.strip() in low for alt in expected.split("|"))
    if check_type == "all_keywords":
        return all(term.strip() in low for term in expected.split(";"))
    if check_type == "word_count":
        return len(output.split()) == int(expected)
    if check_type == "numeric":
        found = re.findall(r"-?\d+(?:\.\d+)?", output.replace(",", ""))
        return expected in found
    raise ValueError(f"unknown check_type: {check_type}")


_cases = [
    ("17 times 4 is 68.", "numeric", "68", True),
    ("The answer is 1,800", "numeric", "1800", True),
    ("Sydney is the capital.", "keyword", "canberra", False),
    ("Red, blue and yellow.", "all_keywords", "red;blue;yellow", True),
    ("Here are exactly three words", "word_count", "3", False),
    ("Yes.", "exact", "yes", True),
]
assert all(score_probe(o, c, e) == want for o, c, e, want in _cases)
print("probe scorer self-test passed")

In [ ]:
# Copied verbatim from the earlier notebooks so every condition decodes identically.
# do_sample=False keeps every measurement deterministic.
def build_messages(sentence, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": sentence})
    return messages


def generate_response(model, tokenizer, sentence, system_prompt=None,
                      max_new_tokens=MAX_NEW_TOKENS):
    messages = build_messages(sentence, system_prompt)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    prompt_length = inputs["input_ids"].shape[1]
    model.eval()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = output[0, prompt_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def run_probe(model, probe_frame):
    """Generate an answer for every probe item and score it."""
    outputs = [generate_response(model, tokenizer, p) for p in probe_frame["prompt"]]
    correct = [
        score_probe(out, row.check_type, str(row.expected))
        for out, row in zip(outputs, probe_frame.itertuples())
    ]
    return outputs, correct

In [ ]:
# STEP 1: calibrate the probe against the BASE model, and take every base-model
# measurement NOW. PeftModel.from_pretrained later edits base_model in place, so
# after that point the name `base_model` no longer means the unmodified model.
#
# Calibration discipline from last module: an item the base already fails has no
# headroom to fall. Drop it, and say what you dropped.
base_probe_outputs_all, base_probe_correct_all = run_probe(base_model, probe_df)

probe_df["base_correct"] = base_probe_correct_all
probe_df["base_output"] = base_probe_outputs_all

dropped = probe_df.loc[~probe_df["base_correct"], ["category", "prompt", "base_output"]]
probe = probe_df[probe_df["base_correct"]].reset_index(drop=True)

print(f"Base model answered {len(probe)}/{len(probe_df)} probe items correctly.")
print(f"Keeping {len(probe)} calibrated items; dropping {len(dropped)}.\n")
if len(dropped):
    print("Dropped (no headroom to show degradation):")
    display(dropped)

assert len(probe) >= 15, (
    f"Only {len(probe)} usable probe items. Too few to resolve an effect - "
    "add easier items to phi3_capability_probe.csv and re-run."
)

In [ ]:
bertscore = evaluate.load("bertscore")


def score_style(predictions):
    """Mean BERTScore F1 and corpus chrF against the clean Yoda references."""
    scores = bertscore.compute(
        predictions=predictions,
        references=style_references,
        lang="en",
        model_type="distilbert-base-uncased",
    )
    return float(np.mean(scores["f1"])), float(corpus_chrf(predictions, [style_references]).score)


# Base style, measured before any adapter exists. The external references contain no
# interjections, which is exactly why they can referee this experiment.
base_style_plain = [generate_response(base_model, tokenizer, s) for s in style_sentences]
base_bert_plain, base_chrf_plain = score_style(base_style_plain)

print(f"Base, no prompt - BERTScore {base_bert_plain:.4f}  chrF {base_chrf_plain:.2f}")
print(f"Base interjection rate: {interjection_rate(base_style_plain):.1%}")
print(f"Base capability: 1.000 by construction after calibration")

## The pairs

No generation is needed to build them. The dataset itself carries the contrast:
`translation` is clean Yoda, and `translation_extra` usually adds an interjection
that the adapter encountered during SFT. The calibration cells verify the intended
contrast and report any pair that contains additional wording changes.

Almost. The next two cells apply the same calibration discipline as the probe, then
measure the one systematic difference we did NOT intend.

Here, **chosen** means preferred for one narrow, operational goal; it does not mean
universally correct or better. Real preference labels can encode annotator disagreement,
cultural assumptions, and inconsistent standards. Treat the labels as measurements to
audit, not ground truth.

In [ ]:
# STEP 2: build and calibrate the preference pairs.
yoda = load_dataset("dvgodoy/yoda_sentences", split="train").to_pandas()

# Leakage guard: no training sentence may appear in the held-out style test.
overlap = set(s.lower().strip() for s in yoda["sentence"]) & \
          set(s.lower().strip() for s in style_sentences)
assert not overlap, f"Held-out sentences found in the training set: {overlap}"

pairs = pd.DataFrame({
    "sentence": yoda["sentence"],
    "chosen_text": yoda["translation"],
    "rejected_text": yoda["translation_extra"],
})

# A pair only carries signal if the checker separates its two sides. Keep a pair when
# chosen is clean, rejected shows the habit, and the texts differ. Report the rest.
ok_chosen = ~pairs["chosen_text"].apply(has_interjection)
ok_rejected = pairs["rejected_text"].apply(has_interjection)
ok_differ = pairs["chosen_text"] != pairs["rejected_text"]
keep = ok_chosen & ok_rejected & ok_differ

print(f"chosen side already clean:     {int(ok_chosen.sum())}/{len(pairs)}")
print(f"rejected side shows the habit: {int(ok_rejected.sum())}/{len(pairs)}")
print(f"sides differ:                  {int(ok_differ.sum())}/{len(pairs)}")

dropped_pairs = pairs[~keep]
pairs = pairs[keep].reset_index(drop=True)
print(f"\nKeeping {len(pairs)} pairs; dropping {len(dropped_pairs)}.")
if len(dropped_pairs):
    display(dropped_pairs.head(10))

assert len(pairs) >= 300, (
    "Too few contrastive pairs to train on. Check the interjection pattern against "
    "what translation_extra actually contains."
)

In [ ]:
# Potential confounds, measured BEFORE training.
#
# DPO learns every systematic difference between chosen and rejected, not just the one
# you intended. Interjections add words, so the rejected side is longer. "Drop the
# interjections" and "say less" are entangled in this data, and no flag can untangle
# them. All we can do is measure how big the entanglement is and check the output
# lengths again after training. We also check whether removing marker tokens and
# punctuation leaves the same words on both sides of each retained pair.
def marker_free_content(text):
    return _norm(INTERJECTION_RE.sub("", text))

marker_only = np.array([
    marker_free_content(chosen) == marker_free_content(rejected)
    for chosen, rejected in zip(pairs["chosen_text"], pairs["rejected_text"])
])
extra_contrast = pairs.loc[~marker_only]
print(f"pairs differing only by designated markers/punctuation: "
      f"{int(marker_only.sum())}/{len(pairs)}")
if len(extra_contrast):
    print("Pairs with additional wording differences (another measured confound):")
    display(extra_contrast)

chosen_words = pairs["chosen_text"].str.split().str.len()
rejected_words = pairs["rejected_text"].str.split().str.len()

print(f"chosen   mean length: {chosen_words.mean():.1f} words")
print(f"rejected mean length: {rejected_words.mean():.1f} words")
print(f"mean gap:             {(rejected_words - chosen_words).mean():.1f} words")
print(f"pairs where rejected is longer: {(rejected_words > chosen_words).mean():.1%}")
print("\nRemember this gap when you read the final table. If mean output length falls")
print("by clearly more than the interjections alone explain, part of what the model")
print("learned was 'be brief', which nobody asked for. The pairs taught it anyway.")

In [ ]:
# Conversational format, the same shape SFTTrainer used in Chapter 0. TRL applies the
# chat template itself. A slice of the pairs is held out for the margin check: pairs
# the trainer never sees, so they can referee whether the preference generalizes.
def to_dpo_format(row):
    return {
        "prompt": [{"role": "user", "content": row["sentence"]}],
        "chosen": [{"role": "assistant", "content": row["chosen_text"]}],
        "rejected": [{"role": "assistant", "content": row["rejected_text"]}],
    }


records = [to_dpo_format(r) for _, r in pairs.iterrows()]

rng = np.random.default_rng(42)
idx = rng.permutation(len(records))
heldout_records = [records[i] for i in idx[:HELDOUT_PAIRS]]
train_records = [records[i] for i in idx[HELDOUT_PAIRS:]]
train_pairs = Dataset.from_list(train_records)

print(f"training pairs: {len(train_pairs)}   held out for the margin check: {len(heldout_records)}")

## One base model, two copies of your adapter

DPO needs two models. The **policy** is the model being trained. The **reference** is a
frozen anchor, and the loss prices how far the policy drifts from it. That price is
`beta`: high beta holds the policy close to the anchor, low beta lets it wander.

Which anchor matters. If the reference were the base model, the loss would charge the
policy for being Yoda at all. We want to keep the style and drop the tic, so the anchor
is the SFT model itself. With adapters this costs no extra memory: load the same
adapter twice under two names. One trains. One never moves.

For prompt $x$, chosen response $y_w$, and rejected response $y_l$, the default
DPO loss for one pair is

$$-\log \sigma\!\left(\beta\left[\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}\right]\right).$$

The bracket is the policy's chosen-versus-rejected advantage relative to the frozen
reference. Initially both adapters are identical, so the bracket is zero and the loss
is $-\log \sigma(0) = \ln 2 \approx 0.693$. This is why the training log has a
meaningful baseline instead of merely displaying an unexplained number.

In [ ]:
# STEP 3: load the SFT adapter twice.
#   "dpo"       - the policy. Training updates it.
#   "reference" - the frozen anchor. Beta prices divergence from it.
base_model = prepare_model_for_kbit_training(base_model)

model_ft = PeftModel.from_pretrained(base_model, ADAPTER_PATH,
                                     adapter_name="dpo", is_trainable=True)
model_ft.load_adapter(ADAPTER_PATH, adapter_name="reference")
model_ft.set_adapter("dpo")

trainable_parms, tot_parms = model_ft.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable_parms/1e6:.2f}M of {tot_parms/1e6:.0f}M "
      f"({100*trainable_parms/tot_parms:.2f}%)")

In [ ]:
# STEP 4: validate before trusting anything downstream.
#
# Before training, "dpo" and "reference" are byte-identical copies of the same
# adapter, so they must generate identical text. If they do not, an adapter name or
# path is wrong, and every number below would be measuring the wrong thing. A red
# cell here is cheaper than a wrong conclusion later.
check_sentences = style_sentences[:3]

model_ft.set_adapter("reference")
ref_out = [generate_response(model_ft, tokenizer, s) for s in check_sentences]

model_ft.set_adapter("dpo")
dpo_out = [generate_response(model_ft, tokenizer, s) for s in check_sentences]

for a, b in zip(dpo_out, ref_out):
    if a != b:
        print(f"  MISMATCH\n    dpo:       {a!r}\n    reference: {b!r}")

assert dpo_out == ref_out, (
    "The two adapter copies disagree before training. Check ADAPTER_PATH and the "
    "adapter names before going any further."
)
print("Identity check passed: policy and reference start as the same model.")

In [ ]:
# STEP 5: the before picture, measured under the frozen reference adapter. This is
# your SFT model exactly as the last two notebooks knew it.
model_ft.set_adapter("reference")

sft_style = [generate_response(model_ft, tokenizer, s) for s in style_sentences]
sft_bert, sft_chrf = score_style(sft_style)
sft_interj = interjection_rate(sft_style)

sft_probe_outputs, sft_probe_correct = run_probe(model_ft, probe)
sft_capability = float(np.mean(sft_probe_correct))

print(f"SFT adapter - chrF {sft_chrf:.2f}  BERTScore {sft_bert:.4f}")
print(f"SFT adapter - interjection rate {sft_interj:.1%}  (the habit, quantified)")
print(f"SFT adapter - capability {sft_capability:.3f}  (base is 1.000 by construction)")

if sft_interj < 0.2:
    print("\nNOTE: your adapter shows the habit on fewer than 20% of held-out")
    print("sentences. DPO has little to remove here. Expect small deltas and read")
    print("the null-result note at the end before drawing conclusions.")

## Train

Launch the cell and leave it. Expect 10 to 20 minutes on a T4, which is the window for
the paired-judgment activity. While it runs, watch two numbers in the log:
`rewards/margins` should climb above zero, and `loss` should fall from about 0.693
(that is ln 2, the loss when the policy and reference agree on everything).

In [ ]:
# STEP 6: DPO. Everything except beta and the learning rate is the memory recipe
# from Chapter 0: micro-batches, gradient accumulation, checkpointing.
model_ft.set_adapter("dpo")
model_ft.config.use_cache = False

dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    beta=BETA,
    model_adapter_name="dpo",
    ref_adapter_name="reference",
    learning_rate=1e-5,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_prompt_length=64,
    max_length=192,
    logging_steps=10,
    report_to="none",
)

trainer = DPOTrainer(
    model=model_ft,
    args=dpo_config,
    train_dataset=train_pairs,
    processing_class=tokenizer,
)

trainer.train()

model_ft.config.use_cache = True
trainer.save_model(DPO_OUTPUT_DIR)
print(f"\nDPO adapter saved to {DPO_OUTPUT_DIR}")

In [ ]:
## **Attention 344** - copy the trained DPO adapter to your Drive when done
#!cp -r '/content/phi3-mini-yoda-dpo-adapter' '/content/drive/MyDrive/344/code' 

In [ ]:
# STEP 7: evaluate preference learning on held-out pairs the trainer never saw.
#
# DPO's implicit reward for a response y is  beta * (logp_policy(y) - logp_ref(y)).
# On a held-out pair, the margin  reward(chosen) - reward(rejected)  should be positive
# if the model learned a preference that extends beyond the training pairs. It is an
# outcome, not a validity gate: a non-positive result is reported as a null. Before
# training the margin is zero by construction because policy and reference are identical.

def completion_logprob(model, prompt_msgs, full_msgs):
    """Sum log-probability of the completion tokens under the chat template."""
    prompt_ids = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True)["input_ids"].to(model.device)
    full_ids = tokenizer.apply_chat_template(
        full_msgs, tokenize=True, return_tensors="pt", return_dict=True,
    )["input_ids"].to(model.device)

    n_prompt = prompt_ids.shape[1]
    model.eval()
    with torch.inference_mode():
        logits = model(full_ids).logits

    logps = torch.log_softmax(logits[0, n_prompt - 1:-1].float(), dim=-1)
    targets = full_ids[0, n_prompt:]
    return float(logps[torch.arange(len(targets)), targets].sum())


def pair_margin(record):
    lp = {}
    for name in ["dpo", "reference"]:
        model_ft.set_adapter(name)
        for side in ["chosen", "rejected"]:
            lp[(name, side)] = completion_logprob(
                model_ft, record["prompt"], record["prompt"] + record[side])
    reward_chosen = BETA * (lp[("dpo", "chosen")] - lp[("reference", "chosen")])
    reward_rejected = BETA * (lp[("dpo", "rejected")] - lp[("reference", "rejected")])
    return reward_chosen - reward_rejected


margins = np.array([pair_margin(r) for r in heldout_records])
print(f"held-out margin: mean {margins.mean():+.3f}   "
      f"positive on {(margins > 0).mean():.0%} of {len(margins)} pairs")

if margins.mean() > 0:
    print("The held-out margin is positive; the learned preference extends beyond the training pairs.")
else:
    print("No positive held-out margin was detected. Treat this as an interpretable null result,")
    print("inspect several pairs and the training loss, and continue to the behavioral table.")

In [ ]:
# STEP 8: the after picture, under the trained "dpo" adapter, decoded exactly like
# every measurement before it.
model_ft.set_adapter("dpo")

dpo_style = [generate_response(model_ft, tokenizer, s) for s in style_sentences]
dpo_bert, dpo_chrf = score_style(dpo_style)
dpo_interj = interjection_rate(dpo_style)

dpo_probe_outputs, dpo_probe_correct = run_probe(model_ft, probe)
dpo_capability = float(np.mean(dpo_probe_correct))

print(f"DPO adapter - chrF {dpo_chrf:.2f}  BERTScore {dpo_bert:.4f}")
print(f"DPO adapter - interjection rate {dpo_interj:.1%}")
print(f"DPO adapter - capability {dpo_capability:.3f}")

In [ ]:
def mean_words(texts):
    return float(np.mean([len(t.split()) for t in texts]))


results = pd.DataFrame([
    {"model": "Base Phi-3", "chrF": base_chrf_plain,
     "bertscore_f1": base_bert_plain,
     "interjection_rate": interjection_rate(base_style_plain),
     "capability": 1.0, "mean_words": mean_words(base_style_plain)},
    {"model": "SFT adapter (before)", "chrF": sft_chrf, "bertscore_f1": sft_bert,
     "interjection_rate": sft_interj, "capability": sft_capability,
     "mean_words": mean_words(sft_style)},
    {"model": "DPO adapter (after)", "chrF": dpo_chrf, "bertscore_f1": dpo_bert,
     "interjection_rate": dpo_interj, "capability": dpo_capability,
     "mean_words": mean_words(dpo_style)},
])
display(results.round(3))

print(f"\nHabit:      {sft_interj:.1%} -> {dpo_interj:.1%}")
print(f"Style:      chrF {sft_chrf:.1f} -> {dpo_chrf:.1f}  (references contain no interjections)")
print(f"Capability: {sft_capability:.3f} -> {dpo_capability:.3f}  (the DPO tax, if any)")
print(f"Length:     {mean_words(sft_style):.1f} -> {mean_words(dpo_style):.1f} words  (the confound check)")

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
results.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved to {RESULTS_PATH}")

In [ ]:
# The qualitative payoff: the same sentences, before and after, side by side.
pd.set_option("display.max_colwidth", None)
sample = pd.DataFrame({
    "sentence": style_sentences[:5],
    "SFT (before)": sft_style[:5],
    "DPO (after)": dpo_style[:5],
})
display(sample)

In [ ]:
# OPTIONAL, if time allows: is the DPO adapter still listening to instructions?
# Same prompt-sensitivity test as last module: how much does each model move when the
# Yoda system prompt is added? Roughly 40 generations, a few minutes on a T4.
model_ft.set_adapter("dpo")
dpo_style_prompted = [
    generate_response(model_ft, tokenizer, s, system_prompt=YODA_SYSTEM_PROMPT)
    for s in style_sentences
]
_, dpo_chrf_prompted = score_style(dpo_style_prompted)

print(f"DPO adapter chrF, no prompt:   {dpo_chrf:.2f}")
print(f"DPO adapter chrF, Yoda prompt: {dpo_chrf_prompted:.2f}")
print(f"delta: {dpo_chrf_prompted - dpo_chrf:+.2f}  "
      "(compare with the base and SFT deltas from the alignment-tax notebook)")

## Reading the result

Three things, read together and in this order.

1. **Did the habit go?** The interjection rate is the target metric. If it fell, DPO
   did its job. If it barely moved, see the note below.
2. **What did the fix cost?** Two possible prices. Style: chrF against the clean
   references should hold or rise, because the references never contained the
   interjections. Capability: the probe says whether preference training taxed the
   model the way SFT did. A drop here is DPO's own alignment tax.
3. **What else changed?** The length column is the confound check. The pair-building
   cell told you how many words the interjections account for. If mean output length
   fell by clearly more than that, the model also learned "be brief". Nobody asked
   for that. The pairs taught it, because it was in them.

One caution on giving DPO credit: the SFT model was trained toward interjection-laden
targets and is scored here against clean references, so part of any chrF gain is the
mechanical removal of tic tokens. The interjection rate is the honest headline number.
chrF and BERTScore corroborate that the style survived.

## If the habit barely moved

A real result, not a failure, and it will be read as a success if it is honestly
obtained and clearly reported. Possible reasons: your adapter expresses the habit
weakly, beta = 0.1 holds the policy too close to the reference, or one epoch over
roughly 700 short pairs is too light a touch. The homework's beta sweep is exactly
this investigation. Do not re-run with a different seed hoping for movement.

## Where this sits

SFT taught by demonstration, and the model learned everything in the demonstrations,
tic included. DPO taught by contrast, and the model learned everything in the
contrast, length confound included. Every training signal you can afford is a proxy
for what you actually want. The craft is choosing a proxy whose failures you can
measure, and then measuring them.

**Extension:** sweep `BETA` and justify a setting using all four outcome columns.

*Credits: this module builds on Daniel Voigt Godoy's Chapter 0 QLoRA fine-tuning
notebook and his `dvgodoy/yoda_sentences` dataset.*